# 00 · Colab timing: how long is one epoch on a free T4?

**Owner:** Experimentation Engineer · **Phase 2**

The Phase 3 studies need many runs, and a free Colab session is limited. This notebook measures
**seconds per training epoch** for the full train split (**34,633 images, batch 32 → 1,083 steps**) on a free
Colab **T4**, for:

| # | Model | Image size | Weight storage |
|---|---|---|---|
| 1 | ResNet-50, last block (conv5) unfrozen | 224 px | fp32 |
| 2 | ResNet-50, last block unfrozen | 224 px | bf16_sr |
| 3 | ResNet-50, last block unfrozen | 160 px | bf16_sr |
| 4 | Model A (design in `report/model_a_design.md`) | 224 px | fp32 |

**Method.** Real data through the team pipeline (`src.data.load_plantvillage`, augmentation on), so decoding and
augmentation are part of the time. For each setting: 10 warm-up steps (graph tracing, cuDNN autotuning), then
100 timed steps; the time per step × 1,083 gives the epoch time. One pass over the validation split (8,659 images)
is timed the same way. The models are built here only for timing; the real builders come in `src/models.py`.

## 1. Runtime and repository

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/saperaa/frostleaf.git
%cd frostleaf
!git log --oneline -1
!pip install -q -r requirements.txt

name, memory.total [MiB]
Tesla T4, 15360 MiB
/content/frostleaf
5b1cb09 (HEAD -> main, origin/main, origin/HEAD) Merge pull request #8 from saperaa/feature/experiment-logger
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
numba 0.64.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.3 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


## 2. Data: PlantVillage colour with the fixed leaf-grouped split

In [2]:
!python data/download.py --variants color --skip-plantdoc

plant_village.py: 100% 7.95k/7.95k [00:00<00:00, 2.54MB/s]
leaf-map.json: 100% 2.43M/2.43M [00:00<00:00, 56.0MB/s]
data.zip: 100% 2.18G/2.18G [00:21<00:00, 103MB/s]
color_train.txt: 100% 4.15M/4.15M [00:00<00:00, 71.9MB/s]
Generating train split: 43596 examples [00:09, 4391.72 examples/s]
color_test.txt: 100% 1.02M/1.02M [00:00<00:00, 30.8MB/s]
Generating train split: 10709 examples [00:02, 4888.68 examples/s]
Saving the dataset (2/2 shards): 100% 43596/43596 [00:02<00:00, 19559.58 examples/s]
Saving the dataset (1/1 shards): 100% 10709/10709 [00:00<00:00, 15959.46 examples/s]
saved color: DatasetDict({
    train: Dataset({
        features: ['image', 'image_path', 'label', 'crop', 'disease', 'leaf_id'],
        num_rows: 43596
    })
    test: Dataset({
        features: ['image', 'image_path', 'label', 'crop', 'disease', 'leaf_id'],
        num_rows: 10709
    })
})


In [3]:
import time, math, numpy as np, keras, tensorflow as tf
from src.config import load_config
from src.data import load_plantvillage
from src.precision import PrecisionModel

print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "|", tf.config.list_physical_devices("GPU"))
BATCH, TRAIN_IMAGES, VAL_IMAGES = 32, 34_633, 8_659
STEPS_PER_EPOCH = math.ceil(TRAIN_IMAGES / BATCH)   # 1,083
VAL_STEPS = math.ceil(VAL_IMAGES / BATCH)           # 271
WARMUP, TIMED = 10, 100

def datasets(image_size):
    cfg = load_config("base.yaml")
    cfg["data"]["image_size"] = image_size
    cfg["train"]["batch_size"] = BATCH
    train_ds, val_ds, _, class_names = load_plantvillage("color", cfg)
    return train_ds, val_ds, class_names

TensorFlow 2.21.0 | Keras 3.15.1 | [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 3. The models (timing versions)

In [4]:
def resnet50_last_block(image_size, precision, num_classes=38):
    keras.utils.set_random_seed(42)
    base = keras.applications.ResNet50(include_top=False, weights="imagenet",
                                       input_shape=(image_size, image_size, 3))
    for layer in base.layers:   # only conv5 trains; BatchNorm stays frozen
        layer.trainable = layer.name.startswith("conv5_") and not isinstance(layer, keras.layers.BatchNormalization)
    inputs = keras.Input((image_size, image_size, 3))
    x = keras.applications.resnet50.preprocess_input(inputs * 255.0)   # pipeline gives [0, 1]
    x = base(x, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax")(x)
    return PrecisionModel(inputs, outputs, precision=precision, rounding_seed=42)

def model_a(image_size, precision, num_classes=38):
    keras.utils.set_random_seed(42)
    inputs = keras.Input((image_size, image_size, 3))
    x = inputs   # report/model_a_design.md rescales 0-255 inside; the pipeline already gives [0, 1]
    for filters in (32, 64, 128, 256):
        x = keras.layers.Conv2D(filters, 3, padding="same", kernel_initializer="he_normal")(x)
        x = keras.layers.BatchNormalization()(x)
        x = keras.layers.ReLU()(x)
        x = keras.layers.MaxPooling2D()(x)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax")(x)
    return PrecisionModel(inputs, outputs, precision=precision, rounding_seed=42)

## 4. Timing

In [6]:
class StepTimer(keras.callbacks.Callback):
    def on_train_batch_end(self, batch, logs=None):
        if batch == WARMUP - 1:
            self.t0 = time.perf_counter()
        if batch == WARMUP + TIMED - 1:
            self.seconds = time.perf_counter() - self.t0

def time_setting(name, builder, image_size, precision, lr):
    train_ds, val_ds, _ = datasets(image_size)
    model = builder(image_size, precision)
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", jit_compile=False)  # XLA fails on PrecisionModel's step counter (CPU variable) on GPU
    model.quantize_weights()
    timer = StepTimer()
    model.fit(train_ds.take(WARMUP + TIMED), epochs=1, verbose=0, callbacks=[timer])
    per_step = timer.seconds / TIMED
    model.evaluate(val_ds.take(WARMUP), verbose=0)                    # warm up evaluation
    t0 = time.perf_counter(); model.evaluate(val_ds.take(50), verbose=0)
    val_per_step = (time.perf_counter() - t0) / 50
    epoch_s = per_step * STEPS_PER_EPOCH + val_per_step * VAL_STEPS
    row = dict(setting=name, size=image_size, precision=precision,
               trainable=int(sum(np.prod(v.shape) for v in model.trainable_variables)),
               s_per_step=round(per_step, 4), train_s=round(per_step * STEPS_PER_EPOCH),
               val_s=round(val_per_step * VAL_STEPS), epoch_s=round(epoch_s),
               epochs_in_30min=int(1800 // epoch_s))
    print(row)
    keras.backend.clear_session()
    return row

results = []
results.append(time_setting("ResNet-50 last block", resnet50_last_block, 224, "fp32", 1e-5))
results.append(time_setting("ResNet-50 last block", resnet50_last_block, 224, "bf16_sr", 1e-5))
results.append(time_setting("ResNet-50 last block", resnet50_last_block, 160, "bf16_sr", 1e-5))
results.append(time_setting("Model A", model_a, 224, "fp32", 1e-3))

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


{'setting': 'ResNet-50 last block', 'size': 224, 'precision': 'fp32', 'trainable': 15031334, 's_per_step': 0.2734, 'train_s': 296, 'val_s': 41, 'epoch_s': 337, 'epochs_in_30min': 5}


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


{'setting': 'ResNet-50 last block', 'size': 224, 'precision': 'bf16_sr', 'trainable': 15031334, 's_per_step': 0.3095, 'train_s': 335, 'val_s': 38, 'epoch_s': 374, 'epochs_in_30min': 4}


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


{'setting': 'ResNet-50 last block', 'size': 160, 'precision': 'bf16_sr', 'trainable': 15031334, 's_per_step': 0.1783, 'train_s': 193, 'val_s': 23, 'epoch_s': 216, 'epochs_in_30min': 8}


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


{'setting': 'Model A', 'size': 224, 'precision': 'fp32', 'trainable': 399142, 's_per_step': 0.2721, 'train_s': 295, 'val_s': 11, 'epoch_s': 306, 'epochs_in_30min': 5}


## 5. Results

In [7]:
import pandas as pd
table = pd.DataFrame(results)
table

,setting,size,precision,trainable,s_per_step,train_s,val_s,epoch_s,epochs_in_30min
0,ResNet-50 last block,224,fp32,15031334,0.2734,296,41,337,5
1,ResNet-50 last block,224,bf16_sr,15031334,0.3095,335,38,374,4
2,ResNet-50 last block,160,bf16_sr,15031334,0.1783,193,23,216,8
3,Model A,224,fp32,399142,0.2721,295,11,306,5


## 6. Recommendation

The cell below states it from the measured numbers: the image size to use and the largest number of epochs
that fits a 30-minute Colab budget per run (training + one validation pass per epoch).

In [8]:
c224, c224sr, c160sr, a224 = results
print(f"Model C (ResNet-50, bf16_sr): 224 px = {c224sr['epoch_s']} s/epoch, 160 px = {c160sr['epoch_s']} s/epoch "
      f"({c224sr['epoch_s'] / c160sr['epoch_s']:.2f}x faster at 160 px).")
print(f"bf16_sr vs fp32 at 224 px: {c224sr['epoch_s'] / c224['epoch_s']:.2f}x the fp32 time (the simulated rounding costs this much).")
size = 224 if c224sr["epochs_in_30min"] >= 5 else 160
chosen = c224sr if size == 224 else c160sr
print(f"Recommendation: image size {size} px; at most {chosen['epochs_in_30min']} epochs per 30-minute run for Model C "
      f"(bf16_sr), {c224['epochs_in_30min']} for Model C in fp32 at 224 px, {a224['epochs_in_30min']} for Model A at 224 px.")

Model C (ResNet-50, bf16_sr): 224 px = 374 s/epoch, 160 px = 216 s/epoch (1.73x faster at 160 px).
bf16_sr vs fp32 at 224 px: 1.11x the fp32 time (the simulated rounding costs this much).
Recommendation: image size 160 px; at most 8 epochs per 30-minute run for Model C (bf16_sr), 5 for Model C in fp32 at 224 px, 5 for Model A at 224 px.


## 7. Conclusions (T4, 26 Nov 2026)

| Setting | s/step | train s | val s | **s/epoch** | epochs in 30 min |
|---|---|---|---|---|---|
| ResNet-50 last block, 224 px, fp32 | 0.273 | 296 | 41 | **337** | 5 |
| ResNet-50 last block, 224 px, bf16_sr | 0.310 | 335 | 38 | **374** | 4 |
| ResNet-50 last block, 160 px, bf16_sr | 0.178 | 193 | 23 | **216** | 8 |
| Model A, 224 px, fp32 | 0.272 | 295 | 11 | **306** | 5 |

**Recommendation: train at 160 px.** Model C (bf16_sr) then takes 216 s per epoch, so a 30-minute run fits **8 epochs**; at 224 px it fits only 4. bf16_sr costs 1.11x the fp32 time at 224 px.

**The input pipeline is the bottleneck, not the GPU.** Model A (0.4 M parameters) and ResNet-50 (15.0 M trainable) take the same time per step (0.272 vs 0.273 s), and 160 px is 1.73x faster than 224 px (close to the 1.96x pixel ratio). Caching decoded images or prefetching in `src/data.py` would speed up every run.

**GPU finding for `src/precision.py`.** On a GPU, Keras compiles the train step with XLA by default, and `PrecisionModel` then fails: its step counter (`self._step`, a `tf.Variable` created in `__init__`, line 159) is placed on the CPU and XLA refuses it ("Trying to access resource rounding_step ... located in device CPU:0 from device GPU:0"). This notebook compiles with `jit_compile=False` for all four settings, so the comparison stays fair. Until `precision.py` is changed (e.g. creating the counter with `self.add_weight` or `keras.Variable`), `train.py` must compile with `jit_compile=False` on Colab.
